# Milestone 1

This milestone focuses on understanding the dataset and establishing a baseline performance through **exploratory data analysis (EDA)** and simple **heuristic-based methods** using `librosa`.

---

## Suggested Readings
- [Hugging Face Audio Course](https://huggingface.co/learn/audio-course/en/chapter0/introduction)
- [Librosa Documentation](https://librosa.org/doc/main/core.html#audio-loading)

---

## Instructions
Use this notebook to answer **all Milestone-1 questions**.

---

## Resources
- Notebook Link:  
  https://colab.research.google.com/drive/1m6UczhxQIke_raWSqukSWuiKbIVt7MMb?usp=sharing  

- Competition Link:  
  https://www.kaggle.com/competitions/jan-2026-dl-gen-ai-project/


In [21]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [22]:
#----------------------------- DON'T CHANGE THIS --------------------------
DATA_SEED = 67
TRAINING_SEED = 1234
SR = 22050
DURATION = 5.0
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
TOP_DB=20
TARGET_SNR_DB = 10

random.seed(DATA_SEED)
np.random.seed(DATA_SEED)
torch.manual_seed(DATA_SEED)
torch.cuda.manual_seed(DATA_SEED)

In [23]:
# CONFIGURATION
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
STEMS = {
    "drums.wav": "drums",
    "vocals.wav": "vocals",
    "bass.wav": "bass",
    "other.wav": "other"
}

STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
GENRE_TO_TEST = 'rock'
SONG_INDEX = 0

In [24]:
def build_dataset(root_dir, val_split=0.17, seed=42):

    train_dataset = {g: {v: [] for v in STEM_KEYS} for g in GENRES}
    val_dataset   = {g: {v: [] for v in STEM_KEYS} for g in GENRES}

    rng = random.Random(seed)

    for genre in GENRES:
        genre_path = os.path.join(root_dir, genre)

        if not os.path.exists(genre_path):
            continue

        valid_songs = []

        for song in os.listdir(genre_path):
            song_path = os.path.join(genre_path, song)

            if not os.path.isdir(song_path):
                continue

            stem_paths = {}
            complete = True

            for stem_file, stem_key in STEMS.items():
                file_path = os.path.join(song_path, stem_file)

                if not os.path.exists(file_path):
                    complete = False
                    break

                if os.path.getsize(file_path) < 4 * 1024:
                    complete = False
                    break

                stem_paths[stem_key] = file_path

            if complete:
                valid_songs.append(stem_paths)

        rng.shuffle(valid_songs)

        split_idx = int(len(valid_songs) * (1 - val_split))
        train_songs = valid_songs[:split_idx]
        val_songs   = valid_songs[split_idx:]

        for song in train_songs:
            for key in STEM_KEYS:
                train_dataset[genre][key].append(song[key])

        for song in val_songs:
            for key in STEM_KEYS:
                val_dataset[genre][key].append(song[key])

    return train_dataset, val_dataset

tr, val = build_dataset(DATA_ROOT)

In [38]:
def find_long_silences(dataset_dict, sr=SR, threshold_sec=DURATION, top_db=TOP_DB):
    """
    Input:
        dataset_dict: The dictionary structure {genre: {stem: [paths...]}}
    Output:
        df: Pandas DataFrame containing details of all files with silence >= 5s
    """
    records = []
    # ------------------- write your code here -------------------------------

    # ---- COUNT TOTAL FILES ----
    total_files = sum(len(paths) for genre_data in dataset_dict.values() 
                     for paths in genre_data.values())
    
    print(f"Analyzing {total_files} files for long silences...")
    
    # Create progress bar
    pbar = tqdm(total=total_files, desc="Processing files")
    
    for genre, stems_dict in dataset_dict.items():
        for stem_name, file_paths in stems_dict.items():
            for file_path in file_paths:
                try:
                    # Load Audio
                    y, _ = librosa.load(file_path, sr=sr, duration=None)
                    total_duration = len(y) / sr
                    
                    # Find Non-Silent Intervals
                    non_silent_intervals = librosa.effects.split(y, top_db=top_db)
                    
                    max_silence = 0.0
                    silence_type = []
                    
                    if len(non_silent_intervals) == 0:
                        # CASE A: Fully silent
                        max_silence = total_duration
                        silence_type = ["start", "middle", "end"]
                    else:
                        # CASE B: START silence
                        start_silence = non_silent_intervals[0][0] / sr
                        if start_silence > max_silence:
                            max_silence = start_silence
                            silence_type = ["start"]
                        
                        # CASE C: END silence
                        end_silence = (len(y) - non_silent_intervals[-1][1]) / sr
                        if end_silence > max_silence:
                            max_silence = end_silence
                            silence_type = ["end"]
                        elif abs(end_silence - max_silence) < 0.01:  # Same as current max
                            if "end" not in silence_type:
                                silence_type.append("end")
                        
                        # CASE D: MIDDLE silence
                        for i in range(len(non_silent_intervals) - 1):
                            gap_start = non_silent_intervals[i][1]
                            gap_end = non_silent_intervals[i + 1][0]
                            gap_duration = (gap_end - gap_start) / sr
                            
                            if gap_duration > max_silence:
                                max_silence = gap_duration
                                silence_type = ["middle"]
                            elif abs(gap_duration - max_silence) < 0.01:
                                if "middle" not in silence_type:
                                    silence_type.append("middle")
                    
                    # Store result
                    if max_silence >= threshold_sec:
                        records.append({
                            "Genre": genre,
                            "Stem": stem_name,
                            "Duration": round(total_duration, 2),
                            "Max_Silence_Sec": round(max_silence, 2),
                            "Silence_Location": ", ".join(silence_type),
                            "File_Path": file_path
                        })
                
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                
                pbar.update(1)
    
    pbar.close()
    #-------------------------------------------------------------------------
    df = pd.DataFrame(records)
    return df


# --- EXECUTION ---
# Pass your 'tr' (training) dictionary here.
# Ensure 'tr' is defined from your previous build_dataset code.
df_silence = find_long_silences(tr, threshold_sec=DURATION, top_db=TOP_DB)

# --- RESULTS ANALYSIS ---

# ------------------- write your code here -------------------------------
print("\n" + "="*60)
print("SILENCE ANALYSIS RESULTS")
print("="*60)
print(f"\nTotal files with silence >= {DURATION}s: {len(df_silence)}")
print(f"\nDataFrame shape: {df_silence.shape}")
print("\nFirst few rows:")
print(df_silence.head())
#-------------------------------------------------------------------------
# Create a pivot Table: Count by Genre vs Stem
print("\n" + "="*60)
print("PIVOT TABLE: Count by Genre vs Stem")
print("="*60)
if len(df_silence) > 0:
    pivot_table = df_silence.pivot_table(index='Genre', columns='Stem', aggfunc='size', fill_value=0)
    print(pivot_table)
else:
    print("No files with long silences found.")

Analyzing 3320 files for long silences...


Processing files: 100%|██████████| 3320/3320 [05:36<00:00,  9.87it/s]


SILENCE ANALYSIS RESULTS

Total files with silence >= 5.0s: 678

DataFrame shape: (678, 6)

First few rows:
   Genre   Stem  Duration  Max_Silence_Sec Silence_Location  \
0  blues  drums     30.01            16.28           middle   
1  blues  drums     30.01             6.34           middle   
2  blues  drums     30.01             6.11           middle   
3  blues  drums     30.01             5.76           middle   
4  blues  drums     30.01             5.29           middle   

                                           File_Path  
0  /kaggle/input/jan-2026-dl-gen-ai-project/messy...  
1  /kaggle/input/jan-2026-dl-gen-ai-project/messy...  
2  /kaggle/input/jan-2026-dl-gen-ai-project/messy...  
3  /kaggle/input/jan-2026-dl-gen-ai-project/messy...  
4  /kaggle/input/jan-2026-dl-gen-ai-project/messy...  

PIVOT TABLE: Count by Genre vs Stem
Stem       bass  drums  other  vocals
Genre                                
blues        17     25      6      45
classical    66     57      4  

In [27]:
stems_audio = []

for key in STEM_KEYS:
    file_path = tr[GENRE_TO_TEST][key][SONG_INDEX]
    audio, _ = librosa.load(file_path, sr=SR, duration=DURATION)
    stems_audio.append(audio)

print("Audio loaded successfully.")

Audio loaded successfully.


In [29]:
stems_stack = np.stack(stems_audio, axis=0)

mix_raw = np.sum(stems_stack, axis=0)

rms_val = np.sqrt(np.mean(mix_raw ** 2))

max_val = np.max(np.abs(mix_raw))

if max_val > 0:
    mix_norm = mix_raw / max_val
else:
    mix_norm = mix_raw

assert np.isclose(np.max(np.abs(mix_norm)), 1.0), "Normalization failed."

# What is the value of total number of corrupted sounds ( less than 4kb) + (total number of sounds < 5.0491MB)?

In [30]:
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

corrupt_threshold = 4 * 1024
size_threshold = 5.0491 * 1024 * 1024

corrupt_count = 0
small_count = 0
total_files = 0

for genre in os.listdir(DATA_ROOT):
    genre_path = os.path.join(DATA_ROOT, genre)

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)

        for file in os.listdir(song_path):
            if file.endswith(".wav"):
                total_files += 1
                file_path = os.path.join(song_path, file)
                size = os.path.getsize(file_path)

                if size < corrupt_threshold:
                    corrupt_count += 1

                if size < size_threshold:
                    small_count += 1

print("Total files:", total_files)
print("Corrupted (<4KB):", corrupt_count)
print("Smaller than 5.0491MB:", small_count)
print("Final Answer:", corrupt_count + small_count)

Total files: 4000
Corrupted (<4KB): 0
Smaller than 5.0491MB: 1256
Final Answer: 1256


# What is the absolute difference between  total number of sounds > 5.0493MB and total number of sounds < 5.0491MB ?

In [31]:
DATA_ROOT = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

lower_threshold = 5.0491 * 1024 * 1024
upper_threshold = 5.0493 * 1024 * 1024

count_greater = 0
count_smaller = 0

for genre in os.listdir(DATA_ROOT):
    genre_path = os.path.join(DATA_ROOT, genre)

    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)

        for file in os.listdir(song_path):
            if file.endswith(".wav"):
                file_path = os.path.join(song_path, file)
                size = os.path.getsize(file_path)

                if size > upper_threshold:
                    count_greater += 1

                if size < lower_threshold:
                    count_smaller += 1

print("Count > 5.0493MB:", count_greater)
print("Count < 5.0491MB:", count_smaller)
print("Absolute Difference:", abs(count_greater - count_smaller))

Count > 5.0493MB: 184
Count < 5.0491MB: 1256
Absolute Difference: 1072


# What is the absolute difference between the number of training reggae drum samples and the number of validation country vocal samples?

In [32]:
train_reggae_drums = len(tr['reggae']['drums'])
val_country_vocals = len(val['country']['vocals'])

print("Training reggae drums:", train_reggae_drums)
print("Validation country vocals:", val_country_vocals)
print("Absolute Difference:", abs(train_reggae_drums - val_country_vocals))

Training reggae drums: 83
Validation country vocals: 17
Absolute Difference: 66


# What's the average Silence Length in Vocals (in secs)?

In [39]:
avg_vocal_silence = df_silence[df_silence["Stem"] == "vocals"]["Max_Silence_Sec"].mean()

print("Average Silence Length in Vocals (secs):", avg_vocal_silence)

Average Silence Length in Vocals (secs): 12.778825396825397


# Total number of drums sound tracks in jazz where silence >= 5 secs and Silence_Location is only middle.

In [40]:
jazz_drums_middle_only = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Silence_Location"] == "middle")
]

print("Total jazz drums with silence >=5s only in middle:",
      len(jazz_drums_middle_only))

Total jazz drums with silence >=5s only in middle: 14


# Total number of drums sound tracks in jazz where silence >= 5 secs and Max_Silence_Sec >= 10.

In [41]:
jazz_drums_long = df_silence[
    (df_silence["Genre"] == "jazz") &
    (df_silence["Stem"] == "drums") &
    (df_silence["Max_Silence_Sec"] >= 10)
]

print("Total jazz drums with silence >=10 sec:",
      len(jazz_drums_long))

Total jazz drums with silence >=10 sec: 7
